# exp5 — mixed-replay ×40 (bases exp2 and exp4, `nonone` variant)

Fine-tuning of two LeukemiaAttri models (Drive folder `LLD/`) on a **mixture** of the
source domain (LeukemiaAttri) and the target domain (ASTER prototype, ×40), to adapt
localization without forgetting the source domain.

| base | ×40-test mAP50 before |
|---|---|
| `exp2_multisharpness__nonone` | 0.396 |
| `exp4_domain_aug__nonone` | 0.371 |

**Goal** (see `localizer_adaptation/README.md`): rise clearly above 0.396 on the
104 held-out ×40 fields, with a target recall ≥ 0.6, while keeping the
LeukemiaAttri test mAP50 ≥ 0.92.

**The notebook measures before and after** for each base, on both domains.

---

### What must be on the Drive

```
MyDrive/LLD/                      the 6 lld_cloud.tar.part-*, run.py, state/
MyDrive/aster_x40dataset/         images/{train,val,test} + labels/{train,val,test}
```

`Runtime → Change runtime type → GPU` before starting.


## 1. Settings

`REPLAY_RATIO` sets the mixture: number of LeukemiaAttri images drawn per ×40 image.
At 4, a batch contains on average 20 % ×40 images — enough to adapt, little enough
not to erase the source domain. The draw is reproducible (seed 42).

`LR0` is deliberately low: training starts from weights that are already good, and a
high rate would destroy them.

`HSV_H = 0.5` is the strong hue augmentation required by the protocol: it forces the
model to decide on morphology rather than on absolute colour, since the prototype
slide is over-stained.

### Why `IMGSZ = 960`

The ×40 images are 4032×3040, the LeukemiaAttri images 640×640. Relative to the
image width, a cell occupies 6.8 % on the ×40 side against 9.7 % on the LeukemiaAttri
side: **×40 cells appear 0.70 times smaller**, and this ratio does not depend on
`IMGSZ`, since both domains undergo the same resizing.

| `IMGSZ` | LeukemiaAttri cell | ×40 cell | ratio |
|---|---|---|---|
| 640 | 62 px | 43.5 px | 0.70 |
| 960 | 93 px | 65.2 px | 0.70 |
| 1280 | 124 px | 87.0 px | 0.70 |

`IMGSZ` therefore does not correct the scale gap — `scale=0.5` does, by varying the
scale from 0.5 to 1.5×, which covers 0.70 comfortably.

The real benefit of 960 lies elsewhere: **preserving the detail of the target
domain**. At 640, a 4032-px-wide image is reduced by a factor of 6.3; at 960, by a
factor of 4.2. The nuclear texture — precisely what distinguishes a leukocyte from a
cluster of red cells on an over-stained slide — survives much better. The
LeukemiaAttri images are enlarged from 640 to 960: no information is gained, but none
is lost either, and the model relearns at this scale.

The cost is 2.25× more computation per image. With only 1 020 images per epoch, this
is affordable.

**No loss of comparability**: the notebook measures before and after at the same
`IMGSZ`, so the reference is recomputed in the same regime.

If the ×40 recall stays below 0.6, the next lever is not a larger `IMGSZ` — it is to
tile the ×40 images at their native resolution.


In [ ]:
BASES = ["exp2_multisharpness__nonone", "exp4_domain_aug__nonone"]

EPOCHS       = 60
PATIENCE     = 10
IMGSZ        = 960
BATCH        = 16
LR0          = 0.001      # low: training starts from already-trained weights
REPLAY_RATIO = 4          # LeukemiaAttri (LLD) images drawn per x40 image
SEED         = 0
WORKERS      = 8

# augmentation: strong hue jitter to ignore the absolute staining colour
AUG = dict(hsv_h=0.5, hsv_s=0.7, hsv_v=0.4,
           scale=0.5, degrees=15.0, mosaic=1.0, fliplr=0.5, flipud=0.5)

DRIVE  = "/content/drive/MyDrive/LLD"
ASTER_DRIVE = "/content/drive/MyDrive/aster_x40dataset"
ROOT   = "/content/lld_cloud"
ASTER  = "/content/aster_x40_dataset"
STATE  = DRIVE + "/state"
RUNS   = ROOT + "/runs"
DEVICE = "0"

print("bases:", BASES)
print("mixture: 1 x40 image per", REPLAY_RATIO, "LLD images | lr0", LR0, "| imgsz", IMGSZ)

## 2. Drive and GPU


In [ ]:
import os, sys, glob, subprocess, shutil, json, random
from google.colab import drive

drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE")
assert gpu, "start a GPU runtime first"
assert os.path.isdir(DRIVE), DRIVE + " not found"

if not os.path.isdir(ASTER_DRIVE):
    autres = [p for p in glob.glob("/content/drive/MyDrive/aster*") if os.path.isdir(p)]
    assert autres, "aster folder not found in MyDrive"
    ASTER_DRIVE = autres[0]
    print("aster folder found under another name:", ASTER_DRIVE)
print("aster on Drive:", ASTER_DRIVE)

## 3. Preparation

LeukemiaAttri is extracted from the archive and the ×40 dataset is copied from the
Drive to the local disk — 762 MB, allow 2 to 5 minutes. Both must be local: reading
25 768 + 351 files through FUSE would make the dataloader unusable.


In [ ]:
import time

if os.path.isdir(ROOT + "/data/images"):
    print("LLD already extracted")
else:
    t0 = time.time()
    !cat {DRIVE}/lld_cloud.tar.part-* | tar --warning=no-unknown-keyword -xf - -C /content/
    print("LLD extracted in %.1f min" % ((time.time() - t0) / 60))

if os.path.isdir(ASTER + "/images/train"):
    print("x40 already copied")
else:
    t0 = time.time()
    shutil.copytree(ASTER_DRIVE, ASTER, dirs_exist_ok=True)
    print("x40 copied in %.1f min" % ((time.time() - t0) / 60))

# possible stray macOS files
for p in glob.glob(ASTER + "/**/._*", recursive=True):
    os.remove(p)

for s in ("train", "val", "test"):
    ni = len(glob.glob(ASTER + "/images/%s/*.jpg" % s))
    nl = len(glob.glob(ASTER + "/labels/%s/*.txt" % s))
    print("x40 %-6s : %3d images / %3d labels" % (s, ni, nl))
    assert ni > 0 and ni == nl, "images and labels out of sync in " + s

cls = set()
for f in glob.glob(ASTER + "/labels/*/*.txt"):
    for l in open(f):
        p = l.split()
        if len(p) == 5:
            cls.add(p[0])
assert cls == {"0"}, "unexpected class ids on the x40 side: " + str(cls)
print("x40: all boxes in class 0")

In [ ]:
!pip -q install ultralytics
import ultralytics; ultralytics.checks()

## 4. Previous state

Restores the base weights and detects incomplete fine-tunes.


In [ ]:
import csv

os.makedirs(STATE, exist_ok=True)
if os.path.isdir(STATE + "/runs"):
    shutil.copytree(STATE + "/runs", RUNS, dirs_exist_ok=True)


def etat(dossier):
    f = dossier + "/results.csv"
    if not os.path.exists(f):
        return False, 0, 0
    L = list(csv.DictReader(open(f)))
    if not L:
        return False, 0, 0
    n = len(L)
    best = int(max(L, key=lambda x: float(x["metrics/mAP50-95(B)"]))["epoch"])
    return (n == best + PATIENCE or n >= EPOCHS), n, best


for b in BASES:
    p = RUNS + "/" + b + "/weights/best.pt"
    assert os.path.exists(p), "missing base weights: " + p
    print("base OK :", b)

for d in sorted(os.listdir(RUNS)) if os.path.isdir(RUNS) else []:
    if not d.startswith("mixedreplay__"):
        continue
    ok, n, best = etat(RUNS + "/" + d)
    if not ok:
        msg = "INCOMPLETE (%d epochs, best %d) -> deleted, will be redone: %s"
        print(msg % (n, best, d))
        shutil.rmtree(RUNS + "/" + d, ignore_errors=True)
        shutil.rmtree(STATE + "/runs/" + d, ignore_errors=True)

faits = [d for d in (os.listdir(RUNS) if os.path.isdir(RUNS) else [])
         if d.startswith("mixedreplay__") and os.path.exists(RUNS + "/" + d + "/weights/best.pt")]
print("\nfine-tunes already done:", sorted(faits) or "(none)")

## 5. Building the mixture

On the LeukemiaAttri side, the labels switch to `nonone` — the variant of both bases.
The replay subset is drawn from the training list **of the original experiment** of
each base: exp2 replays multi-sharpness, exp4 multi-domain. Each one revises what it
learned.

The files produced are lists of absolute paths. Ultralytics derives the label by
replacing `/images/` with `/labels/`, which works on both sides.


In [ ]:
EXP_DE_BASE = {"exp2_multisharpness__nonone": "exp2_multisharpness",
               "exp4_domain_aug__nonone":     "exp4_domain_aug"}

# LeukemiaAttri (LLD) side: labels = nonone (true WBCs only), the variant of both bases
lien = ROOT + "/data/labels"
if os.path.islink(lien) or os.path.exists(lien):
    os.unlink(lien) if os.path.islink(lien) else shutil.rmtree(lien)
os.symlink(ROOT + "/labels_variants/nonone", lien)
print("LLD data/labels ->", os.path.basename(os.path.realpath(lien)))

MIX = "/content/mix"
os.makedirs(MIX, exist_ok=True)


def lld_liste(exp, split):
    src = ROOT + "/lists/%s_%s.txt" % (exp, split)
    return [ROOT + "/" + l for l in open(src).read().split()]


def x40_liste(split):
    return sorted(glob.glob(ASTER + "/images/%s/*.jpg" % split))


def ecrire(chemin, lignes):
    open(chemin, "w").write("\n".join(lignes) + "\n")
    return chemin


yaml_train, yaml_lld_test = {}, {}
for base in BASES:
    exp = EXP_DE_BASE[base]
    rng = random.Random(42)

    x_tr, x_va = x40_liste("train"), x40_liste("val")
    l_tr, l_va = lld_liste(exp, "train"), lld_liste(exp, "val")
    r_tr = rng.sample(l_tr, min(len(l_tr), REPLAY_RATIO * len(x_tr)))
    r_va = rng.sample(l_va, min(len(l_va), REPLAY_RATIO * len(x_va)))

    f_tr = ecrire(MIX + "/%s_train.txt" % base, sorted(r_tr + x_tr))
    f_va = ecrire(MIX + "/%s_val.txt" % base, sorted(r_va + x_va))

    y = MIX + "/%s_mix.yaml" % base
    open(y, "w").write("path: /content\ntrain: %s\nval:   %s\ntest:  %s\n\nnc: 1\nnames:\n  0: wbc\n" %
                         (f_tr, f_va, f_va))
    yaml_train[base] = y

    ylt = MIX + "/%s_lldtest.yaml" % base
    f_te = ecrire(MIX + "/%s_lld_test.txt" % base, lld_liste(exp, "test"))
    open(ylt, "w").write("path: /content\ntrain: %s\nval:   %s\ntest:  %s\n\nnc: 1\nnames:\n  0: wbc\n" %
                           (f_te, f_te, f_te))
    yaml_lld_test[base] = ylt

    print("%-32s train %4d LLD + %3d x40 = %4d | val %3d + %2d" %
            (base, len(r_tr), len(x_tr), len(r_tr) + len(x_tr), len(r_va), len(x_va)))

# x40 test set, identical for both bases: the deployment figure
f_x40te = ecrire(MIX + "/x40_test.txt", x40_liste("test"))
yaml_x40_test = MIX + "/x40_test.yaml"
open(yaml_x40_test, "w").write("path: /content\ntrain: %s\nval:   %s\ntest:  %s\n\nnc: 1\nnames:\n  0: wbc\n" %
                                 (f_x40te, f_x40te, f_x40te))
print("\nx40 test:", len(x40_liste("test")), "fields")

for p in glob.glob(ROOT + "/**/*.cache", recursive=True) + glob.glob(ASTER + "/**/*.cache", recursive=True):
    os.remove(p)
print("label caches cleared")

## 6. "Before" measurement

The base weights evaluated as they are on both test sets. This is the reference
against which the fine-tune will be judged.


In [ ]:
from ultralytics import YOLO
import contextlib, io

P_RES = "/content/mixedreplay_results.json"
res = json.load(open(P_RES)) if os.path.exists(P_RES) else {}
if os.path.exists(STATE + "/mixedreplay_results.json") and not res:
    shutil.copyfile(STATE + "/mixedreplay_results.json", P_RES)
    res = json.load(open(P_RES))


def evalue(poids, ycfg, etiquette):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        r = YOLO(poids).val(data=ycfg, split="val", imgsz=IMGSZ, batch=BATCH,
                            device=DEVICE, classes=[0], plots=False, verbose=False,
                            project="/content/evalmix", name=etiquette, exist_ok=True)
    b = r.box
    return dict(precision=float(b.mp), recall=float(b.mr),
                map50=float(b.map50), map5095=float(b.map))


for base in BASES:
    w = RUNS + "/" + base + "/weights/best.pt"
    for tag, y in (("x40_test", yaml_x40_test), ("lld_test", yaml_lld_test[base])):
        k = "avant|%s|%s" % (base, tag)
        if k in res:
            continue
        res[k] = evalue(w, y, "avant_%s_%s" % (base, tag))
        v = res[k]
        print("before %-32s %-9s P=%.4f R=%.4f mAP50=%.4f mAP50-95=%.4f" %
                (base, tag, v["precision"], v["recall"], v["map50"], v["map5095"]), flush=True)
json.dump(res, open(P_RES, "w"), indent=1)

## 7. Mixed-replay fine-tune


In [ ]:
import threading

_stop = threading.Event()


def sauvegarde():
    try:
        if os.path.isdir(RUNS):
            shutil.copytree(RUNS, STATE + "/runs", dirs_exist_ok=True)
        if os.path.exists(P_RES):
            shutil.copyfile(P_RES, STATE + "/mixedreplay_results.json")
        return True
    except Exception as ex:
        print("[backup] failed:", ex)
        return False


def _boucle():
    while not _stop.wait(900):
        if sauvegarde():
            print("[backup]", time.strftime("%H:%M:%S"), "-> Drive", flush=True)


threading.Thread(target=_boucle, daemon=True).start()
print("automatic backup every 15 min")

In [ ]:
for base in BASES:
    nom = "mixedreplay__" + base
    best = RUNS + "/" + nom + "/weights/best.pt"

    if os.path.exists(best):
        print("[skip]", nom, "already trained")
    else:
        print("\n" + "=" * 72)
        print("[train]", nom, "| %d epochs max, patience %d, lr0 %g" % (EPOCHS, PATIENCE, LR0))
        print("=" * 72, flush=True)
        t0 = time.time()
        m = YOLO(RUNS + "/" + base + "/weights/best.pt")
        m.train(data=yaml_train[base], epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                lr0=LR0, patience=PATIENCE, device=DEVICE, seed=SEED,
                deterministic=True, workers=WORKERS, plots=False, verbose=True,
                project=RUNS, name=nom, exist_ok=True, **AUG)
        print("[done] %s in %.1f min" % (nom, (time.time() - t0) / 60), flush=True)

    for tag, y in (("x40_test", yaml_x40_test), ("lld_test", yaml_lld_test[base])):
        k = "apres|%s|%s" % (base, tag)
        res[k] = evalue(best, y, "apres_%s_%s" % (base, tag))
        v = res[k]
        print("after  %-32s %-9s P=%.4f R=%.4f mAP50=%.4f mAP50-95=%.4f" %
                (base, tag, v["precision"], v["recall"], v["map50"], v["map5095"]), flush=True)
    json.dump(res, open(P_RES, "w"), indent=1)
    sauvegarde()

## 8. Final backup

Run in every case.


In [ ]:
_stop.set()
assert sauvegarde(), "backup failed"
print("state saved in", STATE)

## 9. Results

Success criterion: ×40-test mAP50 clearly above 0.396 with a recall ≥ 0.6, and a
LeukemiaAttri test mAP50 that stays ≥ 0.92.


In [ ]:
entete = ("base", "set", "P before", "P after", "R before", "R after",
          "mAP50 before", "mAP50 after")
print("%-30s%-10s%9s%9s%9s%9s%13s%13s" % entete)
print("=" * 102)
for base in BASES:
    for tag in ("x40_test", "lld_test"):
        a, b = res.get("avant|%s|%s" % (base, tag)), res.get("apres|%s|%s" % (base, tag))
        if not (a and b):
            continue
        print("%-30s%-10s%9.4f%9.4f%9.4f%9.4f%13.4f%13.4f" %
                (base.replace("_nonone", ""), tag, a["precision"], b["precision"],
                 a["recall"], b["recall"], a["map50"], b["map50"]))
    print("-" * 102)

print("\nVerdict")
for base in BASES:
    x = res.get("apres|%s|x40_test" % base)
    l = res.get("apres|%s|lld_test" % base)
    if not (x and l):
        continue
    ok_x = x["map50"] > 0.396 and x["recall"] >= 0.6
    ok_l = l["map50"] >= 0.92
    print("  %-32s x40 mAP50=%.4f R=%.4f %s | LLD mAP50=%.4f %s" %
            (base, x["map50"], x["recall"], "OK" if ok_x else "insufficient",
             l["map50"], "kept" if ok_l else "REGRESSION"))